In [1]:
import numpy as np
import pandas as pd
from scipy.io import mmread

## P2 (a)

In [2]:
# Load the matrix
# bcspwr09     (Power Networks)

A = mmread("/Users/lpwer/Documents/NetSIPhD/PHYS7332_NetData/HW2/power-bcspwr09/power-bcspwr09.mtx").tocoo()

# Save as edge list
with open("edges.csv", "w") as f:
    f.write("Source,Target\n")
    for i, j in zip(A.row, A.col):
        if i != j:  # optional: skip self-loops
            f.write(f"{i},{j}\n")

Modularity: 0.893

Modularity with resolution: 0.893

Number of Communities: 24

In [3]:
df = pd.read_csv('/Users/lpwer/Documents/NetSIPhD/PHYS7332_NetData/HW2/filtered_table.csv')

partition_MM = dict(zip(df['Id'], df['modularity_class']))

## P2 (b)

In [7]:
from graph_tool.all import *
import graph_tool.all as gt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import rc
rc('axes', fc='w')
rc('figure', fc='w')
rc('savefig', fc='w')
rc('axes', axisbelow=True)
import json
from collections import Counter

In [3]:
df_edges = pd.read_csv('/Users/lpwer/Documents/NetSIPhD/PHYS7332_NetData/phys7332_fa25/HW2/edges.csv')
# x = pd.concat([df_edges[ucal], df_edges[vcal]])
# print(x)
labels = pd.Index(pd.unique(pd.concat([df_edges['Source'], df_edges['Target']])))
# print(labels)

label_to_int = {lab: i for i, lab in enumerate(labels)}
# print(labels_to_int)

df_int = df_edges.copy()
df_int['Source'] = df_int['Source'].map(label_to_int)
df_int['Target'] = df_int['Target'].map(label_to_int)

print(df_int)

      Source  Target
0          0     966
1          1     967
2          2     968
3          3     969
4          4     970
...      ...     ...
4783     367     963
4784      42     257
4785     257     964
4786     124     965
4787     575      52

[4788 rows x 2 columns]


In [5]:
g = gt.Graph(directed=False)
g.add_vertex(len(labels))
g.add_edge_list(df_int[['Source', 'Target']].itertuples(index=False, name=None))
gt.remove_self_loops(g)
gt.remove_parallel_edges(g)
pos = gt.sfdp_layout(g)
g.vp.pos = pos

# 1) recreat mapping
int_to_label = {i: lab for lab, i in label_to_int.items()}

name = g.new_vertex_property("string")
for i in range(g.num_vertices()):
    name[g.vertex(i)] = str(int_to_label[i])
g.vp['name'] = name            

# 2) Save positions once and keep them on THIS 'g'
pos = gt.sfdp_layout(g)
g.vp['pos'] = pos

# 3) Community detection (SBM)
state = gt.minimize_blockmodel_dl(g)
blocks = state.get_blocks()
B = state.get_B()

# 4) Colors
cmap = matplotlib.colormaps.get_cmap("tab20")  # continuous colormap
palette = [cmap(i / max(B - 1, 1))[:3] for i in range(B)]
vcolor = g.new_vertex_property("vector<double>")
for v in g.vertices():
    vcolor[v] = palette[int(blocks[v])]

gt.graph_draw(
    g, pos=g.vp['pos'],
    vertex_fill_color=vcolor,
    vertex_size=5, edge_pen_width=0.5,
    output="network_colored_by_community.png"
)

# 5) Export partition
partition_gt = {g.vp['name'][v]: int(blocks[v]) for v in g.vertices()}

import json
with open("partition.json", "w") as f:
    json.dump(partition_gt, f, indent=2)

g.save("network_with_pos_and_name.gt.gz")

In [35]:
# 0. start from existing graph `g` that already has g.vp['name'] set

gt.seed_rng(42)

# 1. degree-preserving randomization (configuration model rewiring)
g_rand = gt.Graph(g)  
# preserves the exact degree sequence; no self-loops / parallel edges
gt.random_rewire(
    g_rand,
    model="configuration",   # degree-preserving
    n_iter=100,               # a few full edge sweeps to mix
    edge_sweep=True,
    parallel_edges=False,
    self_loops=False
)

# quick sanity check: same degree multiset as original
deg_g = np.sort(g.get_total_degrees(g.get_vertices()))
print(deg_g)
deg_g_rand = np.sort(g_rand.get_total_degrees(g_rand.get_vertices()))
print(deg_g_rand)

[ 1  1  1 ... 11 13 14]
[ 1  1  1 ... 11 13 14]


In [36]:
# --- 3) Community detection #1: SBM (degree-corrected) ---
sbm_state = gt.minimize_blockmodel_dl(g_rand)
sbm_blocks = sbm_state.get_blocks()
sbm_partition = {g_rand.vp['name'][v]: int(sbm_blocks[v]) for v in g_rand.vertices()}
with open("partition_randomized_SBM.json", "w") as f:
    json.dump(sbm_partition, f, indent=2)

print("SBM communities:", sbm_state.get_B())

SBM communities: 1723


In [33]:
import community as community_louvain
import networkx as nx

In [37]:
# extract edges from g_rand
edges = [(int(e.source()), int(e.target())) for e in g_rand.edges()]

# build NX graph with same nodes (by index)
Gnx = nx.Graph()
Gnx.add_nodes_from(range(g_rand.num_vertices()))
Gnx.add_edges_from(edges)

# run Louvain
louvain_membership = community_louvain.best_partition(Gnx, random_state=42)

# map indices -> original labels
name = g_rand.vp['name']
louvain_partition = {str(name[g_rand.vertex(i)]): int(comm)
                     for i, comm in louvain_membership.items()}

with open("partition_randomized_Louvain.json", "w") as f:
    json.dump(louvain_partition, f, indent=2)

print("Louvain communities:", len(set(louvain_membership.values())))

Louvain communities: 56


## P2 d